In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import numpy as np

# Step 1: Load the data
data = pd.read_csv("/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/merged_eeg_embeddings.csv")

# Separate features
eeg_features = data.iloc[:, :130].values  # (238, 130)
embeddings = data.iloc[:, 130:].values    # (238, 384) -- check for potential off-by-one issue due to csv index

# Check the type and shape of embeddings
print(f"Embeddings type: {type(embeddings)}")
print(f"Embeddings shape: {embeddings.shape}")

# Ensure embeddings are in the correct type and shape
if embeddings.dtype != np.float32 and embeddings.dtype != np.float64:
    embeddings = embeddings.astype(np.float32)

# Turn both features and embeddings into tensors
eeg_features = torch.tensor(eeg_features, dtype=torch.float32)
embeddings_tensor = torch.tensor(embeddings, dtype=torch.float32)

# Step 2: Define a custom Dataset
class EEGDataset(Dataset):
    def __init__(self, eeg_features, embeddings):
        self.eeg_features = eeg_features
        self.embeddings = embeddings

    def __len__(self):
        return len(self.eeg_features)

    def __getitem__(self, idx):
        return {
            'eeg_features': self.eeg_features[idx],
            'embedding_target': self.embeddings[idx]
        }

dataset = EEGDataset(eeg_features, embeddings_tensor)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Step 3: Define the model
class EEG2TextModel(nn.Module):
    def __init__(self):
        super(EEG2TextModel, self).__init__()
        # Define a simple linear projector
        self.projector = nn.Sequential(
            nn.Linear(130, 384),  # Project EEG features from 130 -> 384
            nn.ReLU(),
            nn.Linear(384, 384)   # Final output for T5 (to match expected embedding size)
        )
        self.t5 = T5ForConditionalGeneration.from_pretrained('t5-small')

    def forward(self, eeg_features, labels=None):
        # Project EEG features to match input embedding size
        projected = self.projector(eeg_features)
        # Expand dims to (batch_size, seq_len=1, hidden_dim) for T5
        projected = projected.unsqueeze(1)
        output = self.t5(inputs_embeds=projected, labels=labels)
        return output

# Instantiate model
model = EEG2TextModel()

# Tokenizer
tokenizer = T5Tokenizer.from_pretrained('t5-small')

# Step 4: Define collate function (important for labels)
def collate_fn(batch):
    eeg_features = torch.stack([item['eeg_features'] for item in batch])

    # Dummy labels: Placeholder text for generation
    dummy_texts = ["Generate text"] * len(batch)  # For training, you'd need actual text targets here
    labels = tokenizer(dummy_texts, padding=True, truncation=True, return_tensors="pt").input_ids

    return {
        'eeg_features': eeg_features,
        'labels': labels
    }

# Step 5: Training arguments
training_args = TrainingArguments(
    output_dir="./eeg2text",
    evaluation_strategy="no",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_strategy="no",
    logging_steps=10,
    remove_unused_columns=False,
)

# Step 6: Custom Trainer
class EEGTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        eeg_features = inputs['eeg_features']
        labels = inputs['labels']
        outputs = model(eeg_features, labels=labels)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

trainer = EEGTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collate_fn,
)

# Step 7: Train
trainer.train()

# Step 8: Inference function
def generate_text_from_eeg(eeg_array):
    model.eval()
    with torch.no_grad():
        eeg_tensor = torch.tensor(eeg_array, dtype=torch.float32).unsqueeze(0)  # Add batch dimension
        projected = model.projector(eeg_tensor).unsqueeze(1)  # Add seq_len dimension for T5
        generated_ids = model.t5.generate(inputs_embeds=projected, max_length=50)
        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return generated_text

# Example: generate text for the first sample
sample_eeg = eeg_features[0]
generated_text = generate_text_from_eeg(sample_eeg)
print(generated_text)


Embeddings type: <class 'numpy.ndarray'>
Embeddings shape: (238, 385)


ValueError: could not convert string to float: 'subject010_Morning'

In [4]:
print(data.columns)

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       ...
       '374', '375', '376', '377', '378', '379', '380', '381', '382', '383'],
      dtype='object', length=515)


In [8]:
# Check data types of each column in the DataFrame
# Check the unique data types of each column
print(data.dtypes.unique())


[dtype('float64') dtype('O')]


In [9]:
# Find columns with 'object' dtype (usually text or mixed data types)
object_columns = data.columns[data.dtypes == 'object']

# Print the column names
print(object_columns)


Index(['filename'], dtype='object')


In [10]:
data = data.drop(columns=object_columns)
# Check the data types of the DataFrame after dropping object columns
print(data.dtypes.unique())
# Check the shape of the DataFrame after dropping object columns
print(data.shape)
# Check the first few rows of the DataFrame after dropping object columns
print(data.head())

[dtype('float64')]
(238, 514)
          0         1         2         3         4         5         6  \
0  1.747556  2.962125  1.988042  3.126621  3.164129  3.042635  3.064530   
1  2.487151  2.857442  2.342438  3.167465  3.247938  3.176121  3.050527   
2  2.798569  2.917469  2.623518  2.996873  2.875600  2.794819  2.972272   
3  2.281045  2.921070  2.135753  2.897236  2.901512  2.881745  2.916120   
4  2.335232  2.248811  2.276023  2.177892  2.196120  2.336399  2.198464   

          7         8         9  ...       374       375       376       377  \
0  2.490361  3.041726  3.079670  ... -0.006144  0.012106 -0.062826 -0.016166   
1  2.516486  3.112483  3.083376  ...  0.056335  0.005741  0.004773 -0.064084   
2  2.581185  2.995557  2.831952  ...  0.022042 -0.038666 -0.038844 -0.011076   
3  2.691121  2.932313  2.902236  ... -0.017982  0.024790 -0.100422  0.003803   
4  2.289123  2.153385  2.422315  ...  0.046634  0.062465  0.029038  0.048842   

        378       379       380       

In [20]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import numpy as np

# Set device to GPU, MPS, or CPU based on availability
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# Step 1: Load the data
data = pd.read_csv("/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/merged_eeg_embeddings.csv")
data = data.select_dtypes(exclude=['object'])  # Drop object columns (non-numeric)

# Separate EEG features and embeddings
eeg_features = data.iloc[:, :130].values  # EEG features are the first 130 columns
embeddings = data.iloc[:, 130:].values    # Embeddings are from column 130 onward

# Check the type and shape of embeddings
print(f"Embeddings type: {type(embeddings)}")
print(f"Embeddings shape: {embeddings.shape}")

# Ensure embeddings are in the correct type and shape
if embeddings.dtype != np.float32 and embeddings.dtype != np.float64:
    embeddings = embeddings.astype(np.float32)

# Convert EEG features and embeddings into tensors and move to the correct device
eeg_features = torch.tensor(eeg_features, dtype=torch.float32).to(device)
embeddings_tensor = torch.tensor(embeddings, dtype=torch.float32).to(device)

# Step 2: Define a custom Dataset
class EEGDataset(Dataset):
    def __init__(self, eeg_features, embeddings):
        self.eeg_features = eeg_features
        self.embeddings = embeddings

    def __len__(self):
        return len(self.eeg_features)

    def __getitem__(self, idx):
        return {
            'eeg_features': self.eeg_features[idx],
            'embedding_target': self.embeddings[idx]
        }

dataset = EEGDataset(eeg_features, embeddings_tensor)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Step 3: Define the model
class EEG2TextModel(nn.Module):
    def __init__(self):
        super(EEG2TextModel, self).__init__()
        # Define a simple linear projector to map EEG features (130) to T5 hidden size (512)
        self.projector = nn.Sequential(
            nn.Linear(130, 512),  # Project EEG features from 130 -> 512 to match T5 hidden size
            nn.ReLU(),
            nn.Linear(512, 512)   # Final output for T5 (to match expected embedding size of 512)
        )
        self.t5 = T5ForConditionalGeneration.from_pretrained('t5-small')

    def forward(self, eeg_features, labels=None):
        # Project EEG features to match input embedding size (512)
        projected = self.projector(eeg_features)
        # Expand dims to (batch_size, seq_len=1, hidden_dim) for T5
        projected = projected.unsqueeze(1)  # Add sequence length dimension
        output = self.t5(inputs_embeds=projected, labels=labels)
        return output


# Instantiate model and move it to the device (GPU, MPS, or CPU)
model = EEG2TextModel().to(device)

# Tokenizer
tokenizer = T5Tokenizer.from_pretrained('t5-small')

# Step 4: Define collate function (important for labels)
def collate_fn(batch):
    eeg_features = torch.stack([item['eeg_features'] for item in batch])

    # Dummy labels: Placeholder text for training (this will be replaced with actual targets in real usage)
    dummy_texts = ["Generate text"] * len(batch)  # For training, you'd need actual text targets here
    labels = tokenizer(dummy_texts, padding=True, truncation=True, return_tensors="pt").input_ids

    return {
        'eeg_features': eeg_features.to(device),  # Move eeg_features to the correct device
        'labels': labels.to(device)  # Move labels to the correct device
    }

# Step 5: Training arguments for older version
training_args = TrainingArguments(
    output_dir="./eeg2text",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_strategy="no",  # Older versions may not support this argument
    logging_dir='./logs',  # logs directory (useful for TensorBoard or debugging)
    remove_unused_columns=False,
)

# Step 6: Custom Trainer
class EEGTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, *args, **kwargs):
        eeg_features = inputs['eeg_features']
        labels = inputs['labels']
        # Call model's forward function to get the loss
        outputs = model(eeg_features, labels=labels)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

trainer = EEGTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collate_fn,
)

# Step 7: Train
trainer.train()

# Step 8: Inference function
def generate_text_from_eeg(eeg_array):
    model.eval()
    with torch.no_grad():
        # Convert EEG features to tensor and move to device
        eeg_tensor = eeg_array.clone().detach().unsqueeze(0).to(device)  # Add batch dimension and move to device
        projected = model.projector(eeg_tensor).unsqueeze(1).to(device)  # Add seq_len dimension for T5 and move to device
        
        # Generate text (e.g., this is a placeholder generation mechanism, you could refine this)
        generated_ids = model.t5.generate(inputs_embeds=projected, max_length=50)
        
        # Decode the generated tokens to text
        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return generated_text

# Example: generate text for the first sample
sample_eeg = eeg_features[0]
generated_text = generate_text_from_eeg(sample_eeg)
print(generated_text)


Embeddings type: <class 'numpy.ndarray'>
Embeddings shape: (238, 384)


Step,Training Loss


Generate text


# trial 

In [31]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch.optim as optim

# Set the device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

eeg_features = pd.read_csv("/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/eeg_features_reordered.csv")
reports_df = pd.read_csv("/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Filtered_Cleaned.csv")
text_reports = reports_df["Cleaned Text"].tolist()  # Ensure it loads as a list of strings

# Convert the data to numeric, forcing non-numeric to NaN and replacing them with 0
eeg_features = eeg_features.apply(pd.to_numeric, errors='coerce').fillna(0)

# Step 2: Tokenizer and Text Model
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model_t5 = T5ForConditionalGeneration.from_pretrained('t5-small')

# Step 3: Define a model to process EEG features and generate text
class EEG2TextModel(nn.Module):
    def __init__(self):
        super(EEG2TextModel, self).__init__()
        # Project EEG features to match the size of T5's hidden states (e.g., 512)
        self.projector = nn.Sequential(
            nn.Linear(131, 512),  # Adjusted to match the input features (131 EEG features -> 512 hidden units)
            nn.ReLU(),
            nn.Linear(512, 512)   # Match final hidden size to T5's hidden states
        )
        
    def forward(self, eeg_features):
        # Pass EEG features through the projector to get a hidden representation
        return self.projector(eeg_features)

# Instantiate the model
model = EEG2TextModel().to(device)

# Step 4: Define custom dataset to load EEG features and text reports
class EEGDataset(Dataset):
    def __init__(self, eeg_features, text_reports):
        self.eeg_features = eeg_features
        self.text_reports = text_reports

    def __len__(self):
        return len(self.eeg_features)

    def __getitem__(self, idx):
        return {
            'eeg_features': self.eeg_features[idx],
            'text_reports': self.text_reports[idx]
        }

# Convert EEG features to tensor and move to the same device
eeg_features_tensor = torch.tensor(eeg_features.values, dtype=torch.float32).to(device)

# Create dataset and dataloader
dataset = EEGDataset(eeg_features_tensor.cpu(), text_reports)  # Keep dataset on CPU for DataLoader compatibility
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Step 5: Define the optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()  # For text generation, CrossEntropyLoss is typically used

# Step 6: Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch in dataloader:
        batch_eeg_features = batch['eeg_features'].to(device)
        batch_text_reports = batch['text_reports']

        optimizer.zero_grad()

        # Forward pass through EEG model to get the hidden representation
        hidden_representation = model(batch_eeg_features)

        # Encode the text reports
        input_ids = tokenizer(batch_text_reports, return_tensors='pt', padding=True, truncation=True, max_length=512).input_ids
        input_ids = input_ids.to(device)  # Ensure input_ids are on the same device

        # Use the hidden representation from EEG model to assist T5 in generating text
        # (Concatenate hidden representation with the input text if needed for generating text)
        # Here you could modify the architecture to use hidden representation as input embedding.

        # Generate text using T5
        outputs = model_t5(input_ids=input_ids, labels=input_ids)

        # Compute loss
        loss = outputs.loss
        running_loss += loss.item()

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(dataloader)}")

# Step 7: Inference function to generate text from EEG data
def generate_text_from_eeg(eeg_data):
    model.eval()
    eeg_tensor = torch.tensor(eeg_data, dtype=torch.float32).unsqueeze(0).to(device, non_blocking=True)
    hidden_representation = model(eeg_tensor)

    # Use the hidden representation as the input to the T5 model for text generation
    input_ids = tokenizer("Generate text", return_tensors='pt').input_ids.to(device)
    
    # Now use the hidden representation to influence text generation (could modify the architecture here)
    # For now, we simply generate text based on a prompt
    generated_ids = model_t5.generate(input_ids=input_ids, max_length=50)

    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return generated_text

# Step 8: Testing on a few samples
num_samples_to_test = 5  # You can adjust this number based on how many samples you want to test

for i in range(num_samples_to_test):
    # Get the i-th sample's EEG data
    sample_eeg = eeg_features_tensor[i].unsqueeze(0)  # Adding batch dimension (1)
    
    # Generate text for the sample EEG data
    generated_text = generate_text_from_eeg(sample_eeg)
    
    # Print the generated text and the corresponding report (ground truth)
    print(f"Sample {i+1}:")
    print(f"Generated Text: {generated_text}")
    print(f"Original Report: {text_reports[i]}")
    print("-" * 80)


Epoch 1/10, Loss: 0.8771073003609975
Epoch 2/10, Loss: 0.8837374895811081
Epoch 3/10, Loss: 0.88957273264726
Epoch 4/10, Loss: 0.8630997478961945
Epoch 5/10, Loss: 0.9027704507112503
Epoch 6/10, Loss: 0.8912247776985168
Epoch 7/10, Loss: 0.8661263565222422
Epoch 8/10, Loss: 0.8602497706810633
Epoch 9/10, Loss: 0.8678563972314198
Epoch 10/10, Loss: 0.8616783062616984
Sample 1:
Generated Text: Generate text
Original Report: I remember having a really hard figuring out where we should go. And , .....I'm just having a long, difficult discussion with my parents, and eating is part of it, but ..I can't remember. It was about someone. It was also about stuff going on with people. It wasn't too. It was kind of serious.
--------------------------------------------------------------------------------
Sample 2:
Generated Text: Generate text
Original Report: I was trying to find different rooms that could be good for me to...and that was a problem where we tried finding me rooms and that was diffi

/var/folders/2h/5twk6xg13mxcmtl80ctk427h0000gn/T/ipykernel_5693/2078181436.py:104: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eeg_tensor = torch.tensor(eeg_data, dtype=torch.float32).unsqueeze(0).to(device, non_blocking=True)


In [30]:
# Step 8: Testing on a few samples
# Choose a few random samples or select the first few samples in the dataset
num_samples_to_test = 5  # You can adjust this number based on how many samples you want to test

for i in range(num_samples_to_test):
    # Get the i-th sample's EEG data
    sample_eeg = eeg_features_tensor[i].unsqueeze(0)  # Adding batch dimension (1)
    
    # Generate text for the sample EEG data
    generated_text = generate_text_from_eeg(sample_eeg)
    
    # Print the generated text and the corresponding report (ground truth)
    print(f"Sample {i+1}:")
    print(f"Generated Text: {generated_text}")
    print(f"Original Report: {text_reports[i]}")
    print("-" * 80)


Sample 1:
Generated Text: Generate text
Original Report: I remember having a really hard figuring out where we should go. And , .....I'm just having a long, difficult discussion with my parents, and eating is part of it, but ..I can't remember. It was about someone. It was also about stuff going on with people. It wasn't too. It was kind of serious.
--------------------------------------------------------------------------------
Sample 2:
Generated Text: Generate text
Original Report: I was trying to find different rooms that could be good for me to...and that was a problem where we tried finding me rooms and that was difficult because theres so many options to pick from and what kind of rooms you . That's it.
--------------------------------------------------------------------------------
Sample 3:
Generated Text: Generate text
Original Report: I'm in an argument with a person and I'm actually describing a dream to the other person about who's done it, and.. they disagree with a decis

/var/folders/2h/5twk6xg13mxcmtl80ctk427h0000gn/T/ipykernel_5693/1895122552.py:100: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eeg_tensor = torch.tensor(eeg_data, dtype=torch.float32).unsqueeze(0).to(device, non_blocking=True)


In [32]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from tqdm import tqdm
import pandas as pd

# === Dataset ===
class EEGDreamDataset(Dataset):
    def __init__(self, eeg_features, dream_texts, tokenizer, max_length=128):
        self.eeg_features = eeg_features
        self.dream_texts = dream_texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.eeg_features)

    def __getitem__(self, idx):
        eeg = torch.tensor(self.eeg_features[idx], dtype=torch.float)
        target = self.tokenizer(
            self.dream_texts[idx],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )
        return eeg, target.input_ids.squeeze(), target.attention_mask.squeeze()

# === Model ===
class EEG2DreamModel(nn.Module):
    def __init__(self, eeg_input_dim, d_model=512, eeg_seq_len=5, pretrained_model_name="t5-small"):
        super(EEG2DreamModel, self).__init__()
        self.d_model = d_model
        self.eeg_seq_len = eeg_seq_len

        # EEG -> Token Sequence Projection
        self.eeg_projection = nn.Linear(eeg_input_dim, d_model * eeg_seq_len)
        self.encoder_proj = nn.Linear(d_model, d_model)

        # T5
        self.text_generator = T5ForConditionalGeneration.from_pretrained(pretrained_model_name)

    def forward(self, eeg_input, decoder_input_ids=None, labels=None):
        batch_size = eeg_input.size(0)

        # EEG -> sequence of embeddings
        projected = self.eeg_projection(eeg_input)
        projected = projected.view(batch_size, self.eeg_seq_len, self.d_model)

        # Project to match encoder
        projected = self.encoder_proj(projected)

        # Run T5 with inputs_embeds
        output = self.text_generator(
            inputs_embeds=projected,
            decoder_input_ids=decoder_input_ids,
            labels=labels,
            return_dict=True
        )
        return output

def generate_dream(model, eeg_input, tokenizer, device, max_length=50):
    model.eval()
    eeg_input = eeg_input.to(device)

    with torch.no_grad():
        projected = model.eeg_projection(eeg_input)
        projected = projected.view(1, model.eeg_seq_len, model.d_model)
        projected = model.encoder_proj(projected)

        generated_ids = model.text_generator.generate(
            inputs_embeds=projected,
            max_length=max_length,
            num_beams=5,
            early_stopping=True,
            forced_bos_token_id=tokenizer.pad_token_id  # Forces a start token
        )

    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# === Training Script ===
def train_model(model, train_loader, optimizer, scheduler, device, num_epochs=5):
    model.train()
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    for epoch in range(num_epochs):
        running_loss = 0.0
        for eegs, input_ids, attention_mask in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            eegs, input_ids, attention_mask = eegs.to(device), input_ids.to(device), attention_mask.to(device)

            outputs = model(eegs, labels=input_ids)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"Average Loss: {avg_loss:.4f}")

# === Main ===
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load the dataset
    eeg_features = pd.read_csv("/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/eeg_features_reordered.csv")
    reports_df = pd.read_csv("/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Filtered_Cleaned.csv")

    # Ensure the EEG features are numeric
    eeg_features = eeg_features.apply(pd.to_numeric, errors='coerce').fillna(0).values

    # The cleaned dream texts (ensure it's a list)
    dream_texts = reports_df["Cleaned Text"].tolist()

    tokenizer = T5Tokenizer.from_pretrained("t5-small")
    dataset = EEGDreamDataset(eeg_features, dream_texts, tokenizer)
    loader = DataLoader(dataset, batch_size=8, shuffle=True)

    model = EEG2DreamModel(eeg_input_dim=eeg_features.shape[1])
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=10,
        num_training_steps=len(loader) * 5
    )

    train_model(model, loader, optimizer, scheduler, device, num_epochs=5)

    # Generate an example
    sample_eeg = torch.randn(1, eeg_features.shape[1])  # Example sample EEG
    generated_dream = generate_dream(model, sample_eeg, tokenizer, device)
    print("\n=== Dream Generation Example ===")
    print(generated_dream)

if __name__ == "__main__":
    main()


Epoch 1/5: 100%|██████████| 30/30 [00:14<00:00,  2.09it/s]


Average Loss: 3.7462


Epoch 2/5: 100%|██████████| 30/30 [00:14<00:00,  2.07it/s]


Average Loss: 1.6554


Epoch 3/5: 100%|██████████| 30/30 [00:14<00:00,  2.02it/s]


Average Loss: 1.5152


Epoch 4/5: 100%|██████████| 30/30 [00:15<00:00,  1.98it/s]


Average Loss: 1.4462


Epoch 5/5: 100%|██████████| 30/30 [00:15<00:00,  1.97it/s]


Average Loss: 1.4153

=== Dream Generation Example ===

